In [ ]:
import yfinance as yf

df = yf.download("1301.TW", start="2023-01-01", end="2023-12-31", auto_adjust=False)
display(df)

In [ ]:
from FinMind.data import DataLoader

df = DataLoader().taiwan_stock_daily(
    stock_id="1301",
    start_date="2023-01-01",
    end_date="2023-12-31",
)
display(df)

In [ ]:
#yfinance_fetcher.py
import pandas as pd
import yfinance as yf

class Yfinance_fetcher:
    def __init__(self, api_key=None):
        self.api_key = api_key

    def fetch(self, ticker, start=None, end=None, period=None):
        # 1. 抓取參數：如果有 start 就不用 period
        params = {
            "tickers": ticker,
            "auto_adjust": False,
            "progress": False, # 關閉進度條讓 console 乾淨點
        }
        
        if start:
            params["start"] = start
            params["end"] = end
        else:
            params["period"] = period or "2y"

        # 2. 執行抓取 (只抓這一次)
        df = yf.download(**params)

        # 3. 檢查是否抓到資料
        if df.empty:
            print(f"警告: 找不到 {ticker} 的資料")
            return pd.DataFrame()

        # 4. 處理 MultiIndex 問題 (針對單一 Ticker 攤平欄位)
        # 有些版本的 yfinance 會在 columns 加上 ticker 名稱
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # 5. 統一清洗資料格式
        df = df.reset_index()
        df.columns = [col.lower().replace(" ", "_") for col in df.columns]
        
        rename_dict = {
            "adj_close": "adj_price"
        }
        df = df.rename(columns=rename_dict)
        
        df['date'] = pd.to_datetime(df['date'])     # 確保 date 欄位是 datetime 格式

        return df

In [ ]:
import pandas as pd
from FinMind.data import DataLoader

class Finmind_fetcher:
    def __init__(self, api_key=None):
        self.api = DataLoader()
        if api_key:
            self.api.login(api_key)

    def fetch(self, ticker, start=None, end=None):
        # 1. 確保日期格式為字串
        start_date = pd.to_datetime(start).strftime('%Y-%m-%d') if start else "2020-01-01"
        end_date = pd.to_datetime(end).strftime('%Y-%m-%d') if end else None

        # 2. 抓取資料
        df = self.api.taiwan_stock_daily(
            stock_id=ticker,
            start_date=start_date,
            end_date=end_date
        )

        if df.empty:
            return pd.DataFrame()

        # 3. 清洗欄位名稱：移除空格、統一轉小寫
        df.columns = df.columns.str.strip().str.lower()

        # 4. 重新命名以對齊 Yfinance 的習慣
        rename_dict = {
            'stock_id': 'ticker',
            'max': 'high',
            'min': 'low',
            'trading_volume': 'volume',
            'trading_money': 'amount',
            'trading_turnover': 'turnover'
        }
        df = df.rename(columns=rename_dict)

        # 5. 確保資料型別正確（FinMind 有時會回傳 Object）
        numeric_cols = ['open', 'high', 'low', 'close', 'volume', 'amount']
        df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
        
        # 確保日期是 datetime 格式並放在欄位中 (不當 index)
        df['date'] = pd.to_datetime(df['date'])

        return df

In [ ]:
import finmind_fetcher

fetcher = finmind_fetcher.Finmind_fetcher()
df = fetcher.fetch("1301", start="2023-01-01", end="2023-12-31")
display(df)

In [ ]:
import yfinance_fetcher

fetcher = yfinance_fetcher.Yfinance_fetcher()
df = fetcher.fetch(ticker="1301.TW, 1101.TW", start="2023-01-01", end="2023-12-31")
display(df)